# Steps
1. Load Data
2. Text Preprocessing
3. Tensors, Datasets, Dataloader
4. RNN
5. Train
6. Evaluate
7. Test by giving new Review

# 1. Load Data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/kaggle/input/datasets/radhikaasmar/imdb-dataset/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.nunique()
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

# 2. Preprocessing

1. Convert to lowercase
2. Remove urls (http, https)
3. Remove Html tags
4. Remove Puntuations
5. Remove Stopwords
6. Stemming
7. Encode sentiment
8. Vectorization (TF-IDF)

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### 1. Converting to Lowercase

In [7]:
df["review"] = df["review"].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 2. Removing the URLs

In [8]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text) # (pattern, replacement, string)
    return text

df["review"] = df["review"].apply(remove_urls)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 3. Remove HTML Tags

In [9]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text) 
    return text

df["review"] = df["review"].apply(remove_html)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 4. Remove Punctuations

In [10]:
def remove_punctuations(text):

    text = re.sub(r"[^A-Za-z0-9\s]", "", text)

    return text

df["review"] = df["review"].apply(remove_punctuations)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Remove Stopwords

In [11]:
import nltk # nature language toolkit

nltk.download("punkt") # tokenizer
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
sample_text = "I like coding in python"
tokens = word_tokenize(sample_text)
tokens

['I', 'like', 'coding', 'in', 'python']

In [14]:
# def remove_stopwords(text):
#     tokens = word_tokenize(text)
#     stop_words = stopwords.words("english")

#     for word in tokens:
#         if word in stop_words:
#             text = text.replace(word, "")

#     return text

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered_tokens = []

    for word in tokens:
        if word not in stop_words:
            filtered_tokens.append(word)

    return " ".join(filtered_tokens)

df["review"] = df["review"].apply(remove_stopwords)
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


### 6. Stemming

In [15]:
# Porter Stemming
from nltk.stem import PorterStemmer

In [16]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


### 7. Encoding

In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]
y.head()

0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

### 8. Vectorization

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4048128 stored elements and shape (49582, 5000)>

# 3. Dataset & DataLoaders

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [20]:
X_train.shape

(34707, 5000)

In [21]:
X_test.shape

(14875, 5000)

In [22]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

# 4. RNN

In [23]:
import torch.nn as nn
import torch.optim as optim

In [27]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        # batch_first=True => requires less computation

        # FC layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        #optional => shape(num of layers, batch size, hidden_state )
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st output: hidden state of all the timestamps => (batch,, seq_len, hidden_state)
        # sequence_len = # of time stamps = no of tokens
        # 2nd output: final hidden state of last timestamp

        out = self.fc(out[:, -1, :])
        return out        

In [28]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# 5. Training the RNN

In [29]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) #(batch_size,1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size, ) => probabilty

        loss = criterion(outputs, yb) #compute loss
        loss.backward() #back-prop
        optimizer.step() #weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.4340316355228424
epoch = 2/10 and loss = 0.3683583438396454
epoch = 3/10 and loss = 0.4635927379131317
epoch = 4/10 and loss = 0.12049132585525513
epoch = 5/10 and loss = 0.3406803607940674
epoch = 6/10 and loss = 0.2872907817363739
epoch = 7/10 and loss = 0.21982163190841675
epoch = 8/10 and loss = 0.2807014584541321
epoch = 9/10 and loss = 0.3553515374660492
epoch = 10/10 and loss = 0.04171628877520561


# 6. Test

In [30]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"acucracy = {correct_vals/total_vals*100}")

acucracy = 86.67563025210085
